# 🧭 Lesson 08: Choosing an Embedding Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/08_choosing_embedding_model.ipynb)

> **The embedding model decides what "similar" means. A model that doesn't speak your language can't find your answers.**

---

## 🎯 Goal
By the end of this lesson you can compare embedding models on your own data, and choose one using evidence instead of habit.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 07](07_embeddings.ipynb) |
| 🔑 **API key** | Not needed (models run locally) |
| 🛠 **Libraries** | `sentence-transformers`, `numpy` |

In [ ]:
!pip install -q -U sentence-transformers numpy

## 🤔 Why does the model matter?

In Lesson 07 we used `all-MiniLM-L6-v2` and it worked well. But models differ in:

| Property | Why it matters |
|---|---|
| **Languages** | A model trained mostly on English may handle Persian badly |
| **Max input length** | Longer text than the limit is silently cut (256 tokens for MiniLM) |
| **Size and speed** | Bigger models are often more accurate but slower and heavier |
| **Vector dimension** | More dimensions means more storage in the vector database |
| **Prefixes** | Some models need a special text in front of queries and passages |

Public benchmarks help, but **your documents and your questions are what count**. So we build a tiny test and measure.

In [1]:
english_chunks = [
    {"section": "NDA", "text": "Employees must protect confidential information belonging to the company, its clients and partners, such as product roadmaps, customer data and proprietary algorithms. It must not be shared with unauthorized people."},
    {"section": "Conduct", "text": "Employees must maintain a respectful, professional environment free from harassment, discrimination and intimidation. Disruptive behavior and verbal abuse are prohibited and may lead to disciplinary action."},
    {"section": "PTO", "text": "Full-time employees receive paid time off: 10 days per year with under 1 year of service, 15 days with 1 to 3 years, and 20 days with more than 3 years. Requests go through the HR system."},
    {"section": "Travel", "text": "Reasonable and necessary expenses of approved business trips are reimbursed. Flights, hotels and meals must be recorded with receipts in the finance system."},
]

english_questions = [
    ("How many vacation days do I get after two years?", "PTO"),
    ("What happens if a coworker insults me?", "Conduct"),
    ("Who pays for my flight to a client meeting?", "Travel"),
    ("Can I share our product plans with a friend?", "NDA"),
]

persian_chunks = [
    {"section": "NDA", "text": "کارکنان باید از اطلاعات محرمانه شرکت، مشتریان و شرکا مانند نقشه راه محصول، داده‌های مشتریان و الگوریتم‌های اختصاصی محافظت کنند. این اطلاعات نباید با افراد غیرمجاز به اشتراک گذاشته شود."},
    {"section": "Conduct", "text": "کارکنان باید محیطی محترمانه و حرفه‌ای و عاری از آزار، تبعیض و ارعاب حفظ کنند. رفتار مخل و توهین کلامی ممنوع است و می‌تواند به اقدام انضباطی منجر شود."},
    {"section": "PTO", "text": "کارکنان تمام‌وقت مرخصی با حقوق دارند: ۱۰ روز در سال برای کمتر از یک سال سابقه، ۱۵ روز برای یک تا سه سال و ۲۰ روز برای بیش از سه سال. درخواست مرخصی از طریق سیستم منابع انسانی ثبت می‌شود."},
    {"section": "Travel", "text": "هزینه‌های معقول و ضروری سفرهای کاری تأییدشده بازپرداخت می‌شود. بلیط پرواز، هتل و غذا باید همراه با رسید در سیستم مالی ثبت شوند."},
]

persian_questions = [
    ("بعد از دو سال کار چند روز مرخصی دارم؟", "PTO"),
    ("اگر همکارم مرا تحقیر کند چه می‌شود؟", "Conduct"),
    ("هزینه بلیط هواپیما برای جلسه مشتری را چه کسی می‌پردازد؟", "Travel"),
    ("آیا می‌توانم برنامه‌های محصول شرکت را با دوستم در میان بگذارم؟", "NDA"),
]

print(len(english_chunks), "English chunks,", len(persian_chunks), "Persian chunks")

4 English chunks, 4 Persian chunks


## 📏 How we compare

For each question we find the best chunk (like in Lesson 07) and check if it is the **correct** one. The share of questions answered correctly is the **top-1 accuracy**.

We compare two models:

| | `all-MiniLM-L6-v2` | `multilingual-e5-small` |
|---|---|---|
| Languages | English-focused | 100+ languages |
| Max input | 256 tokens | 512 tokens |
| Vector size | 384 | 384 |
| Prefixes | none | `query: ` before questions, `passage: ` before chunks |

> ⚠️ **The E5 prefixes are required.** Its authors say that without them you see a performance drop. So we add them for E5 and only for E5.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

models = {
    "MiniLM (English)": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "passage_prefix": "",
    },
    "multilingual-e5-small": {
        "name": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
}

embedders = {model_name: SentenceTransformer(info["name"]) for model_name, info in models.items()}

def evaluate(embedder, info, chunks, questions):
    chunk_vectors = embedder.encode(
        [info["passage_prefix"] + chunk["text"] for chunk in chunks],
        normalize_embeddings=True,
    )
    hits = 0
    for question, correct_section in questions:
        question_vector = embedder.encode([info["query_prefix"] + question], normalize_embeddings=True)[0]
        scores = chunk_vectors @ question_vector
        best = int(np.argmax(scores))
        hits += chunks[best]["section"] == correct_section
    return hits / len(questions)

for language, chunks, questions in [
    ("English", english_chunks, english_questions),
    ("Persian", persian_chunks, persian_questions),
]:
    for model_name, info in models.items():
        accuracy = evaluate(embedders[model_name], info, chunks, questions)
        print(f"{language:8} {model_name:24} top-1 accuracy: {accuracy:.0%}")

Could not reach the Hub ([WinError 10061] No connection could be made because the target machine actively refused it). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5386.90it/s]
'[WinError 10061] No connection could be made because the target machine actively refused it' thrown while requesting GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12733.67it/s]


English  MiniLM (English)         top-1 accuracy: 100%
English  multilingual-e5-small    top-1 accuracy: 100%
Persian  MiniLM (English)         top-1 accuracy: 50%
Persian  multilingual-e5-small    top-1 accuracy: 100%


### 👀 What we saw

| Test | MiniLM (English) | multilingual-e5-small |
|---|---|---|
| English questions | 100% | 100% |
| Persian questions | **50%** | **100%** |

On English, the models tie. On Persian, the English-focused model missed half the questions, while the multilingual model got all of them.

> ⚠️ This is **4 questions per language**. One question changes the result by 25 points. Treat it as a *signal*, not a proof. Real comparisons need more questions (Evaluation lessons).

Let's look at *which* Persian questions MiniLM got wrong:

In [4]:
embedder = embedders["MiniLM (English)"]
chunk_vectors = embedder.encode([chunk["text"] for chunk in persian_chunks], normalize_embeddings=True)

for question, correct_section in persian_questions:
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ question_vector
    best = int(np.argmax(scores))
    mark = "OK  " if persian_chunks[best]["section"] == correct_section else "MISS"
    print(f'{mark} expected={correct_section:8} got={persian_chunks[best]["section"]:8} score={scores[best]:.3f}')
    print("    ", question)

MISS expected=PTO      got=Travel   score=0.696
     بعد از دو سال کار چند روز مرخصی دارم؟
OK   expected=Conduct  got=Conduct  score=0.766
     اگر همکارم مرا تحقیر کند چه می‌شود؟
OK   expected=Travel   got=Travel   score=0.788
     هزینه بلیط هواپیما برای جلسه مشتری را چه کسی می‌پردازد؟
MISS expected=NDA      got=Conduct  score=0.854
     آیا می‌توانم برنامه‌های محصول شرکت را با دوستم در میان بگذارم؟


## 🏷️ Do the E5 prefixes really matter?

The E5 authors say each text should start with `query: ` or `passage: `. Let's test it: we run E5 again **without** prefixes and compare.

In [5]:
no_prefix = {
    "name": models["multilingual-e5-small"]["name"],
    "query_prefix": "",
    "passage_prefix": "",
}

for language, chunks, questions in [
    ("English", english_chunks, english_questions),
    ("Persian", persian_chunks, persian_questions),
]:
    accuracy = evaluate(embedders["multilingual-e5-small"], no_prefix, chunks, questions)
    print(f"{language:8} e5 WITHOUT prefixes top-1 accuracy: {accuracy:.0%}")

English  e5 WITHOUT prefixes top-1 accuracy: 100%
Persian  e5 WITHOUT prefixes top-1 accuracy: 100%


### 👀 What we saw

**MiniLM missed two Persian questions:**

| Question (shortened) | Expected | MiniLM chose | Score |
|---|---|---|---|
| days off after two years | PTO | Travel | 0.696 |
| share product plans with a friend | NDA | Conduct | **0.854** |

Look at the second score: **0.854 for a wrong answer.** In Lesson 07 the *correct* English answers scored only 0.40 to 0.47.

Lesson: **scores can't be compared across models or languages.** A threshold like "above 0.5 means relevant" that works for one model can be meaningless for another. Choose any threshold by measuring, per model and per language.

**The E5 prefixes:** with and without them, E5 scored 100% in both languages. In this tiny test we see **no difference**. That does *not* prove the prefixes are useless: four questions are too few to show it, and the model's authors say to use them. Keep them, and measure on your own data when you can.

> ℹ️ The first run needs an internet connection to download each model. After that, it is loaded from a local cache.

## ✅ A checklist for choosing a model

| Question | Why it matters |
|---|---|
| **Does it speak your language?** | We saw 50% vs 100% on Persian. Check the model card, then test. |
| **Does your chunk fit its max length?** | Text beyond the limit is cut silently (256 tokens for MiniLM, 512 for E5-small). |
| **Which prefixes does it expect?** | Read the model card. Wrong or missing prefixes can hurt quality. |
| **How big and how fast?** | Bigger models are heavier to download and slower to run, often more accurate. |
| **What is the vector dimension?** | More dimensions means more storage in the vector database. |
| **What is the license?** | Check it on the model card before you build on it. |
| **How does it do on *your* data?** | Public benchmarks (like MTEB) help to shortlist. Your own test decides. |

**A simple workflow:** shortlist 2 or 3 models from benchmarks and model cards, build a small test set from *your* documents and *real* questions, compare, then pick.

## 📌 Summary

- The embedding model decides what "similar" means. **Language support matters**: on Persian, the English-focused MiniLM got 50% and multilingual-e5-small got 100% (on our 4-question test).
- Always check the model's **max input length** (256 vs 512 tokens here) and its **prefix** rules (E5 uses `query: ` and `passage: `).
- **Scores are not comparable across models or languages.** We saw a wrong answer with a score of 0.854.
- Compare models on a **small test set built from your own data**. Even 4 questions can reveal a big difference, but they can't prove small ones.
- Always use the **same model** for chunks and questions.

## ✍️ Exercise (10 minutes)

1. Add **two more questions** to `persian_questions` and two to `english_questions` (each with its correct section).
2. Re-run the comparison cell. Did any accuracy change?
3. How many questions do you think you would need before you trusted the numbers? Why?

**Success:** you can explain in one sentence why 4 questions are not enough.

💡 Solution: `solutions/08_choosing_embedding_model.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Break E5 on purpose: use `"passage: "` as the **query** prefix and `"query: "` as the **passage** prefix (swapped).

1. Does the accuracy change on our small test?
2. If not, what does that say about our test set, and not about E5?

Then try a **third model** of your choice from the Hugging Face Hub. Before you run it, read its model card: what languages, max length and prefixes does it have?

## 📖 Next

We have good vectors. Keeping them in a Python variable doesn't scale. In the next lesson we store them in a **vector database** and search them quickly.

**➡️ Next lesson:** [09 Vector Databases](09_vector_databases.ipynb)